In [1]:
import pandas as pd
import glob
import os

# 1. Consolidação Global e Filtro de Performance
# Busca todos os arquivos de ranking gerados pelas 20 batches
files = glob.glob("../experiments_results/inference_ranking/top_30_birds_xgboost_batch_*.csv")

if not files:
    print("❌ Nenhum arquivo encontrado. Verifique o caminho das pastas.")
else:
    # Junta todos os resultados
    df_massive = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    
    # FILTRO: Apenas o que for acima de 0.7
    df_elite_07 = df_massive[df_massive['predicted_F1'] > 0.7].sort_values(by='predicted_F1', ascending=False)
    
    n_encontrados = len(df_elite_07)
    
    print(f"📊 Análise de 1.000.000 de candidatos finalizada.")
    print(f"🔥 Encontrados {n_encontrados} pipelines com F1 > 0.7")
    print("-" * 50)

    if n_encontrados > 0:
        # 2. Identificar colunas de Algoritmos (Flags)
        # Excluímos colunas de parâmetros (com '-') e de predição
        exclude = ['predicted_F1', 'predicted_model_size_log']
        flag_cols = [c for c in df_elite_07.columns if '-' not in c and c not in exclude]
        
        # 3. Contagem de Frequência na Elite
        # Somamos a presença (1) de cada algoritmo
        counts = df_elite_07[flag_cols].sum().sort_values(ascending=False)
        counts = counts[counts > 0] # Mantém apenas os que apareceram pelo menos uma vez
        
        print("🏆 ALGORITMOS MAIS FREQUENTES NA ELITE (> 0.7):")
        print(counts.to_string()) # Exibe a lista completa de quem apareceu
        
        # 4. Salvamento da Elite para análise posterior ou validação no MEKA
        output_path = "../experiments_results/inference_ranking/ELITE_ABOVE_07_BIRDS.csv"
        df_elite_07.to_csv(output_path, index=False)
        print(f"\n✅ Lista de elite salva em: {output_path}")
        
        # 5. Anatomia do Melhor de Todos (Top 1 Global)
        print("\n💎 DETALHES DO MELHOR CANDIDATO (TOP 1):")
        top_1 = df_elite_07.iloc[0]
        print(f"   F1 Previsto: {top_1['predicted_F1']:.4f}")
        
        # Filtra apenas o que está ativo no Top 1
        ativos = top_1[(top_1 != -1.0) & (top_1 != 0.0)]
        for col, val in ativos.items():
            if col not in exclude:
                tipo = "⚙️ Parâmetro" if '-' in col else "✅ Algoritmo"
                print(f"   {tipo}: {col} = {val}")
    else:
        print("⚠️  Nenhum pipeline atingiu 0.7 nesta busca. Tente aumentar o N de instâncias ou revisar as regras da Elite.")

📊 Análise de 1.000.000 de candidatos finalizada.
🔥 Encontrados 9 pipelines com F1 > 0.7
--------------------------------------------------
🏆 ALGORITMOS MAIS FREQUENTES NA ELITE (> 0.7):
feature preprocessing                        9.0
meka.classifiers.multilabel.MULAN.MLkNN      9.0
weka.classifiers.trees.RandomForest          7.0
sklearn.feature_selection.SelectFromModel    2.0
mlfs.br_relieff.BR_ReliefF                   2.0
sklearn.feature_selection.RFE                2.0
weka.classifiers.lazy.KStar                  2.0
mlfs.ppt_skb.PPT_SelectKBest                 1.0
mlfs.lrfs_adapted.PyIT_LRFS                  1.0
mlfs.br_skb.BR_SelectKBest                   1.0

✅ Lista de elite salva em: ../experiments_results/inference_ranking/ELITE_ABOVE_07_BIRDS.csv

💎 DETALHES DO MELHOR CANDIDATO (TOP 1):
   F1 Previsto: 0.7389
   ✅ Algoritmo: feature preprocessing = 1.0
   ✅ Algoritmo: mlfs.br_relieff.BR_ReliefF = 1.0
   ⚙️ Parâmetro: mlfs.br_relieff.BR_ReliefF-n_features = 17.9875400969749